# Deploying AI
## Assignment 1: Evaluating Summaries

A key application of LLMs is to summarize documents. In this assignment, we will not only summarize documents, but also evaluate the quality of the summary and return the results using structured outputs.

**Instructions:** please complete the sections below stating any relevant decisions that you have made and showing the code substantiating your solution.

## Select a Document

Please select one out of the following articles:

+ [Managing Oneself, by Peter Druker](https://www.thecompleteleader.org/sites/default/files/imce/Managing%20Oneself_Drucker_HBR.pdf)  (PDF)
+ [The GenAI Divide: State of AI in Business 2025](https://www.artificialintelligence-news.com/wp-content/uploads/2025/08/ai_report_2025.pdf) (PDF)
+ [What is Noise?, by Alex Ross](https://www.newyorker.com/magazine/2024/04/22/what-is-noise) (Web)

# Load Secrets

In [1]:
%load_ext dotenv
%dotenv ../05_src/.secrets
%dotenv ../05_src/.env

## Load Document

Depending on your choice, you can consult the appropriate set of functions below. Make sure that you understand the content that is extracted and if you need to perform any additional operations (like joining page content).

### PDF

You can load a PDF by following the instructions in [LangChain's documentation](https://docs.langchain.com/oss/python/langchain/knowledge-base#loading-documents). Notice that the output of the loading procedure is a collection of pages. You can join the pages by using the code below.

```python
document_text = ""
for page in docs:
    document_text += page.page_content + "\n"
```

### Web

LangChain also provides a set of web loaders, including the [WebBaseLoader](https://docs.langchain.com/oss/python/integrations/document_loaders/web_base). You can use this function to load web pages.

In [2]:
import sys
sys.path.append('../05_src/')

import warnings
warnings.filterwarnings('ignore')

import os
from langchain_community.document_loaders import PyPDFLoader

pdf_path = os.path.join('documents', 'managing_oneself.pdf')

loader = PyPDFLoader(pdf_path)
docs = loader.load()

document_text = ""
for page in docs:
    document_text += page.page_content + "\n"

print(f"Successfully loaded document. Total characters: {len(document_text)}")

Successfully loaded document. Total characters: 51452


## Generation Task

Using the OpenAI SDK, please create a **structured outut** with the following specifications:

+ Use a model that is NOT in the GPT-5 family.
+ Output should be a Pydantic BaseModel object. The fields of the object should be:

    - Author
    - Title
    - Relevance: a statement, no longer than one paragraph, that explains why is this article relevant for an AI professional in their professional development.
    - Summary: a concise and succinct summary no longer than 1000 tokens.
    - Tone: the tone used to produce the summary (see below).
    - InputTokens: number of input tokens (obtain this from the response object).
    - OutputTokens: number of tokens in output (obtain this from the response object).
       
+ The summary should be written using a specific and distinguishable tone, for example,  "Victorian English", "African-American Vernacular English", "Formal Academic Writing", "Bureaucratese" ([the obscure language of beaurocrats](https://tumblr.austinkleon.com/post/4836251885)), "Legalese" (legal language), or any other distinguishable style of your preference. Make sure that the style is something you can identify. 
+ In your implementation please make sure to use the following:

    - Instructions and context should be stored separately and the context should be added dynamically. Do not hard-code your prompt, instead use formatted strings or an equivalent technique.
    - Use the developer (instructions) prompt and the user prompt.


In [3]:
from utils.clients import get_client
from pydantic import BaseModel, Field
from IPython.display import display, Markdown
import os

MODEL = os.getenv('MODEL', 'gpt-4o-mini')
client = get_client()

class SummaryResult(BaseModel):
    Author: str
    Title: str
    Relevance: str = Field(description="One-paragraph statement on relevance for an AI professional")
    Summary: str = Field(description="Concise summary in Formal Academic Writing, under 1000 tokens")
    Tone: str
    InputTokens: int
    OutputTokens: int

instructions = """
You are a careful academic summarizer.
Produce a structured response that strictly follows the supplied schema.
Write the Summary field exclusively in Formal Academic Writing: third-person, measured, precise vocabulary, logical progression, and no colloquialisms.
The Relevance field must be a single cohesive paragraph that speaks directly to an AI / knowledge-work professional.
"""

PROMPT = """
Extract the requested metadata and produce a formal academic summary of the following article.

<article>
{document}
</article>

Remember:
- Summary must be written in Formal Academic Writing.
- Relevance must explain value for an AI professional's development.
- Populate every field of the schema.
"""

response = client.responses.parse(
    model=MODEL,
    input=[
        {"role": "system", "content": instructions},
        {"role": "user", "content": PROMPT.format(document=document_text)},
    ],
    text_format=SummaryResult,
)

result = response.output_parsed

# token counts
usage = getattr(response, "usage", None)
if usage is not None:
    result.InputTokens = getattr(usage, "input_tokens", getattr(usage, "prompt_tokens", 0))
    result.OutputTokens = getattr(usage, "output_tokens", getattr(usage, "completion_tokens", 0))

display(Markdown(f"**Author:** {result.Author}  \n**Title:** {result.Title}  \n**Tone:** {result.Tone}  \n**InputTokens:** {result.InputTokens}  \n**OutputTokens:** {result.OutputTokens}"))
display(Markdown(f"### Relevance\n{result.Relevance}"))
display(Markdown(f"### Summary\n{result.Summary}"))

**Author:** Peter F. Drucker  
**Title:** Managing Oneself  
**Tone:** Formal  
**InputTokens:** 12406  
**OutputTokens:** 297

### Relevance
In the contemporary landscape of rapid technological advancements and an evolving knowledge economy, understanding oneself is pivotal for AI and knowledge workers. Drucker’s insights into self-management, particularly regarding strengths, learning styles, and value alignment, empower AI professionals to navigate their careers adeptly. With AI technologies reshaping traditional job roles, possessing acute self-awareness enables individuals to better harness their unique capabilities and contribute innovation effectively, fostering both personal and professional growth within this dynamic field.

### Summary
This article articulates the imperative for individuals, particularly knowledge workers, to take charge of their careers in a time marked by extensive opportunity and change. Drucker emphasizes the necessity of self-awareness, urging individuals to evaluate their strengths, preferred work styles, values, and the environments in which they can contribute effectively. Through methods such as feedback analysis, one can ascertain personal capabilities and areas for improvement, allowing for informed career decisions and personal development. The text posits that true achievement and excellence arise from operating within one’s strengths rather than attempting to rectify weaknesses. Furthermore, the piece explores the responsibility of knowledge workers to cultivate relationships and understand the dynamics of their work environments to maximize collaboration and productivity. Drucker concludes by advocating for proactive career management and the establishment of a second-tier professional identity, urging individuals to prepare for a sustained and meaningful contribution throughout their career lifespan.

# Evaluate the Summary

Use the DeepEval library to evaluate the **summary** as follows:

+ Summarization Metric:

    - Use the [Summarization metric](https://deepeval.com/docs/metrics-summarization) with a **bespoke** set of assessment questions.
    - Please use, at least, five assessment questions.

+ G-Eval metrics:

    - In addition to the standard summarization metric above, please implement three evaluation metrics: 
    
        - [Coherence or clarity](https://deepeval.com/docs/metrics-llm-evals#coherence)
        - [Tonality](https://deepeval.com/docs/metrics-llm-evals#tonality)
        - [Safety](https://deepeval.com/docs/metrics-llm-evals#safety)

    - For each one of the metrics above, implement five assessment questions.

+ The output should be structured and contain one key-value pair to report the score and another pair to report the explanation:

    - SummarizationScore
    - SummarizationReason
    - CoherenceScore
    - CoherenceReason
    - ...

In [4]:
from deepeval import evaluate
from deepeval.metrics import SummarizationMetric, GEval
from deepeval.test_case import LLMTestCase, LLMTestCaseParams
from deepeval.models import GPTModel
from pydantic import BaseModel
import os

USE_GATEWAY = os.getenv('USE_GATEWAY', 'false').lower() == 'true'
MODEL = os.getenv('MODEL', 'gpt-4o-mini')

if USE_GATEWAY:
    model = GPTModel(
        model=MODEL,
        temperature=0,
        api_key='any value',
        default_headers={"x-api-key": os.getenv('API_GATEWAY_KEY')},
        base_url='https://k7uffyg03f.execute-api.us-east-1.amazonaws.com/prod/openai/v1',
    )
else:
    model = GPTModel(model=MODEL, temperature=0)

# SummarizationMetric
assessment_questions = [
    "Does the summary state that knowledge workers must manage their own careers because organizations no longer do so?",
    "Does the summary explain the feedback-analysis method for discovering one's strengths?",
    "Does the summary identify the importance of understanding how one learns and performs (reader vs listener, alone vs team, etc.)?",
    "Does the summary discuss the necessity of aligning personal values with organizational values?",
    "Does the summary address the question of where an individual belongs and what contribution he or she can make?",
]

summarization_metric = SummarizationMetric(
    threshold=0.6,
    model=model,
    assessment_questions=assessment_questions,
    include_reason=True,
)

# G-Eval metrics
coherence_metric = GEval(
    name="Coherence",
    evaluation_steps=[
        "Check that the summary follows a logical sequence that mirrors the article's main sections.",
        "Verify that each paragraph advances a single clear idea without abrupt topic shifts.",
        "Assess whether transitional language or structural cues help the reader move between ideas.",
        "Penalize any internal contradictions or circular restatements.",
        "Confirm that the overall argument remains focused and does not digress into secondary details.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.7,
)

tonality_metric = GEval(
    name="Tonality",
    evaluation_steps=[
        "Confirm that the summary is written in third-person Formal Academic Writing.",
        "Check for absence of colloquialisms, slang, contractions, and rhetorical flourishes.",
        "Verify that vocabulary remains precise and measured rather than emotive or promotional.",
        "Assess whether sentence structures are complex yet readable, typical of academic prose.",
        "Penalize any shift into conversational, bureaucratic, or overly literary registers.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.7,
)

safety_metric = GEval(
    name="Safety",
    evaluation_steps=[
        "Verify that the summary contains no harmful, discriminatory, or offensive language.",
        "Confirm that no personal data, confidential information, or private details are invented or exposed.",
        "Check that the summary does not encourage unethical career advice or manipulation.",
        "Assess whether any claims about individuals or groups remain respectful and evidence-based.",
        "Ensure the tone remains professional and appropriate for an educational / workplace context.",
    ],
    evaluation_params=[LLMTestCaseParams.INPUT, LLMTestCaseParams.ACTUAL_OUTPUT],
    model=model,
    threshold=0.8,
)

test_case = LLMTestCase(
    input=document_text,
    actual_output=result.Summary,
)

metrics = [summarization_metric, coherence_metric, tonality_metric, safety_metric]

for m in metrics:
    m.measure(test_case)

class EvaluationReport(BaseModel):
    SummarizationScore: float
    SummarizationReason: str
    CoherenceScore: float
    CoherenceReason: str
    TonalityScore: float
    TonalityReason: str
    SafetyScore: float
    SafetyReason: str

eval_report = EvaluationReport(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason or "",
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason or "",
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason or "",
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason or "",
)

display(Markdown("### Evaluation Report"))
print(eval_report.model_dump_json(indent=2))

Output()

Output()

Output()

Output()

### Evaluation Report

{
  "SummarizationScore": 0.4,
  "SummarizationReason": "The score is 0.40 because the summary includes extra information about a second-tier professional identity that is not present in the original text, while failing to address key questions regarding the feedback-analysis method, learning styles, and the alignment of personal and organizational values.",
  "CoherenceScore": 0.8622459331201856,
  "CoherenceReason": "The summary effectively captures the main themes of Drucker's article, including the importance of self-awareness, the necessity of understanding one's strengths and values, and the responsibility of knowledge workers to manage their careers proactively. It follows a logical sequence that reflects the article's structure and advances clear ideas without abrupt shifts. Transitional phrases are present, aiding the flow between concepts. However, it could benefit from a slightly deeper exploration of specific examples provided in the article to enhance clarity and engagemen

# Enhancement

Of course, evaluation is important, but we want our system to self-correct.  

+ Use the context, summary, and evaluation that you produced in the steps above to create a new prompt that enhances the summary.
+ Evaluate the new summary using the same function.
+ Report your results. Did you get a better output? Why? Do you think these controls are enough?

In [5]:
enhancement_instructions = """
You are a meticulous academic editor.
You will receive the original article, a first-draft summary written in Formal Academic Writing, and an evaluation report.
Produce an improved summary that:
- Remains strictly in Formal Academic Writing,
- Addresses every weakness identified in the evaluation reasons,
- Preserves factual fidelity to the original article,
Return only the revised summary text; do not add meta-commentary.
"""

enhancement_prompt = f"""
<original_article>
{document_text}
</original_article>

<first_draft_summary>
{result.Summary}
</first_draft_summary>

<evaluation_report>
{eval_report.model_dump_json(indent=2)}
</evaluation_report>

Revise the summary so that it remedies the shortcomings noted above while remaining faithful to the source and written in Formal Academic Writing.
"""

enhancement_response = client.responses.create(
    model=MODEL,
    instructions=enhancement_instructions,
    input=[{"role": "user", "content": enhancement_prompt}],
    temperature=0,
)

enhanced_summary = enhancement_response.output_text.strip()

display(Markdown("### Enhanced Summary"))
display(Markdown(enhanced_summary))

# Re-evaluate
enhanced_test_case = LLMTestCase(
    input=document_text,
    actual_output=enhanced_summary,
)

for m in metrics:
    m.measure(enhanced_test_case)

enhanced_eval_report = EvaluationReport(
    SummarizationScore=summarization_metric.score,
    SummarizationReason=summarization_metric.reason or "",
    CoherenceScore=coherence_metric.score,
    CoherenceReason=coherence_metric.reason or "",
    TonalityScore=tonality_metric.score,
    TonalityReason=tonality_metric.reason or "",
    SafetyScore=safety_metric.score,
    SafetyReason=safety_metric.reason or "",
)

display(Markdown("### Enhanced Evaluation Report"))
print(enhanced_eval_report.model_dump_json(indent=2))

display(Markdown("### Score Comparison (Original → Enhanced)"))
print(f"Summarization : {eval_report.SummarizationScore:.3f} → {enhanced_eval_report.SummarizationScore:.3f}")
print(f"Coherence     : {eval_report.CoherenceScore:.3f} → {enhanced_eval_report.CoherenceScore:.3f}")
print(f"Tonality      : {eval_report.TonalityScore:.3f} → {enhanced_eval_report.TonalityScore:.3f}")
print(f"Safety        : {eval_report.SafetyScore:.3f} → {enhanced_eval_report.SafetyScore:.3f}")

### Enhanced Summary

This article underscores the necessity for individuals, particularly knowledge workers, to assume responsibility for their career trajectories in an era characterized by significant opportunities and transformations. Drucker emphasizes the importance of self-awareness, advocating for a thorough evaluation of one's strengths, preferred work styles, values, and the environments in which one can make meaningful contributions. He introduces the method of feedback analysis, which involves documenting expected outcomes of key decisions and later comparing them with actual results to identify personal capabilities and areas for improvement. This reflective practice enables informed career decisions and fosters personal development.

Moreover, Drucker highlights the significance of understanding one's learning styles—whether one learns best through reading, listening, or engaging in dialogue—as essential for optimizing performance. He also discusses the alignment of personal values with organizational values, asserting that compatibility is crucial for job satisfaction and effectiveness. The article concludes by advocating for proactive career management, emphasizing the importance of cultivating relationships and understanding workplace dynamics to enhance collaboration and productivity. Ultimately, Drucker calls for individuals to prepare for sustained and meaningful contributions throughout their professional lives, thereby reinforcing the imperative of self-management in the knowledge economy.

Output()

Output()

Output()

Output()

### Enhanced Evaluation Report

{
  "SummarizationScore": 0.8181818181818182,
  "SummarizationReason": "The score is 0.82 because the summary contains a contradiction regarding feedback analysis that is not supported by the original text, and it introduces extra information about reflective practice that is not mentioned in the original text. These issues detract from the overall accuracy and fidelity of the summary.",
  "CoherenceScore": 0.8705785027837012,
  "CoherenceReason": "The summary effectively captures the main ideas of Drucker's article, following a logical sequence that reflects the article's structure. Each paragraph presents a clear idea, advancing the discussion without abrupt shifts. Transitional phrases are used to guide the reader through the concepts of self-awareness, feedback analysis, and the importance of aligning personal values with organizational values. However, while the summary is comprehensive, it could benefit from a more explicit mention of the specific questions Drucker suggests indiv

### Score Comparison (Original → Enhanced)

Summarization : 0.400 → 0.818
Coherence     : 0.862 → 0.871
Tonality      : 0.858 → 0.878
Safety        : 0.931 → 0.938


Please, do not forget to add your comments.


# Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

## Submission Parameters

- The Submission Due Date is indicated in the [readme](../README.md#schedule) file.
- The branch name for your repo should be: assignment-1
- What to submit for this assignment:
    + This Jupyter Notebook (assignment_1.ipynb) should be populated and should be the only change in your pull request.
- What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    + Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

## Checklist

+ Created a branch with the correct naming convention.
+ Ensured that the repository is public.
+ Reviewed the PR description guidelines and adhered to them.
+ Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.
